In [2]:
from email import policy
from email.parser import BytesParser
from email.utils import parseaddr, parsedate_to_datetime, getaddresses

from html import unescape
from html.parser import HTMLParser

from bs4 import BeautifulSoup
from pathlib import Path

import pandas as pd
import glob
import os

In [3]:
MY_EMAIL = "shend@ibbagency.com"
COLLEAGUE_EMAIL = "besar@ibbagency.com"

In [4]:
PROJECT_ROOT = Path(os.getcwd()).resolve().parents[0]

EMAIL_DIRS = {
    "inbox": PROJECT_ROOT / "emails" / "inbox",
    "sent": PROJECT_ROOT / "emails" / "sent",
    "leads": PROJECT_ROOT / "emails" / "leads",
}

OUTPUT_PATH = PROJECT_ROOT / "data" / "raw" / "emails.csv"

### Helpers

##### HTML Extraction

In [5]:
class HTMLTextExtractor(HTMLParser): # Converts HTML to readable text
    def __init__(self):
        super().__init__()
        self.text = []

    def handle_data(self, data):
        self.text.append(data)

    def get_text(self):
        return " ".join(self.text)


def clean_html(html): # Removes HTML tags
    parser = HTMLTextExtractor()
    parser.feed(html)

    return unescape(" ".join(parser.get_text().split()))

##### E-Mail Extraction

In [6]:
def get_email_addresses(header):
    """
    Extracts email addresses from a header.

    Returns:
        names: Semicolon-separated names
        emails: Semicolon-separated email addresses
    """

    if not header:
        return "", ""

    addresses = getaddresses([str(header)])

    names = []
    emails = []

    for name, address in addresses:
        if name:
            names.append(name)

        if address:
            emails.append(address)

    return "; ".join(names), "; ".join(emails)

In [7]:
def get_email_body(message):
    """
    Extracts the email body.

    Prefers plain text.
    Falls back to HTML if plain text is unavailable.

    Returns:
        body_raw, body_clean
    """

    plain_body = None
    html_body = None

    if message.is_multipart():
        for part in message.walk():

            # Ignore attachments
            if part.get_content_disposition() == "attachment":
                continue

            content_type = part.get_content_type()

            if content_type == "text/plain" and plain_body is None:
                plain_body = part.get_content()

            elif content_type == "text/html" and html_body is None:
                html_body = part.get_content()

    else:
        content_type = message.get_content_type()

        if content_type == "text/plain":
            plain_body = message.get_content()

        elif content_type == "text/html":
            html_body = message.get_content()

    if plain_body:
        body_raw = plain_body
        body_clean = " ".join(plain_body.split())

    elif html_body:
        body_raw = html_body
        body_clean = clean_html(html_body)

    else:
        body_raw = ""
        body_clean = ""

    return body_raw, body_clean


In [8]:
def has_attachments(message):
    """
    Checks whether an email contains attachments.

    Returns:
        True/False
    """

    for part in message.walk():

        if part.get_filename():
            return True

        if part.get_content_disposition() == "attachment":
            return True

    return False

In [9]:
def parse_date(date_header):
    """
    Converts the email date into ISO 8601 format.
    """

    if not date_header:
        return None

    try:
        return parsedate_to_datetime(date_header).isoformat()

    except (TypeError, ValueError, OverflowError):
        return None

##### Parser

In [10]:
def parse_email(file_path, source_folder):
    """
    Parses one .eml file and returns its extracted data.
    """

    with open(file_path, "rb") as file:
        message = BytesParser(policy=policy.default).parse(file)

    # Headers
    sender_name, sender_email = get_email_addresses(
        message.get("From")
    )

    recipient_name, recipient_email = get_email_addresses(
        message.get("To")
    )

    _, cc = get_email_addresses(
        message.get("Cc")
    )

    # Body
    body_raw, body_clean = get_email_body(message)

    # Reply detection
    in_reply_to = message.get("In-Reply-To", "")
    references = message.get("References", "")

    is_reply = bool(in_reply_to or references)

    # Date
    date = parse_date(message.get("Date"))

    # Email direction
    direction = (
        "Sent" if source_folder == "sent" else "Received"
    )

    return {
        # Identification
        "email_id": message.get("Message-ID", ""),

        # Source
        "source_folder": source_folder,
        "folder": source_folder,
        "source_file": str(file_path.relative_to(PROJECT_ROOT)),

        # Date and direction
        "date": date,
        "direction": direction,

        # Sender
        "sender_name": sender_name,
        "sender_email": sender_email,

        # Recipients
        "recipient_name": recipient_name,
        "recipient_email": recipient_email,
        "cc": cc,

        # Subject
        "subject": str(message.get("Subject", "")),

        # Content
        "body_raw": body_raw,
        "body_clean": body_clean,

        # Threading
        "in_reply_to": str(in_reply_to),
        "references": str(references),
        "is_reply": is_reply,

        # Attachments
        "has_attachments": has_attachments(message)
    }

### Parse All E-Mails

In [11]:
def parse_all_emails():
    emails = []

    for folder_name, folder_path in EMAIL_DIRS.items():

        if not folder_path.exists():
            print(f"Directory not found: {folder_path}")
            continue

        eml_files = list(folder_path.rglob("*.eml"))

        print(f"\nProcessing {folder_name}: {len(eml_files)} files")

        for file_path in eml_files:

            try:
                email_data = parse_email(file_path, folder_name)
                emails.append(email_data)

            except Exception as error:
                print(f"Failed to parse {file_path}: {error}")

    return emails

In [12]:
emails = parse_all_emails()


Processing inbox: 250 files

Processing sent: 244 files

Processing leads: 12 files


In [13]:
df = pd.DataFrame(emails)

In [15]:
OUTPUT_PATH

PosixPath('/media/shend/Games_Projets/IMA/sales_intelligence/data/raw/emails.csv')

In [16]:
df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)